## A/B тестирование

### Сценарий эксперимента

#### Проблема  
Продукт продаёт подписки через пейволл (экран оплаты). Текущая конверсия из просмотра пейволла в покупку составляет около 10%. Команда гипотетически теряет пользователей, которые готовы платить, но считают, что базовая цена слишком высокая, или не видят ценности в коротких планах.  
#### Гипотеза  
Если добавить на экран подписки специальное предложение (скидка для годовой подписки), то конверсия в покупку вырастет с 10% до 12.5% (относительный рост +25%), при этом общий ARPU (средняя выручка на пользователя) не упадёт из-за скидки.  
#### Группы эксперимента  
Group A (Control): Стандартный экран оплаты.  
Group B (Treatment): Экран оплаты со скидкой / акцентом на годовой план.  
#### Дерево метрик теста  
**Primary metric:**  
Paywall Conversion Rate = $\frac{количество\ пользователей\ с\ событием\ subscription\_purchase}{количество\ пользователей\ с\ событием\ paywall\_view}$  
**Secondary metric:**  
ARPU = $\frac{Сумма\ completed\_orders}{Общее\ число\ пользователей\ в\ группе}$  
ARPPU = $\frac{Сумма\ completed\_orders}{Число\ платящих\ пользователей\ в\ группе}$  
**Guardrail Metrics**  
D1 / D7 Retention Rate (чтобы убедиться, что новый экран не привлекает "нецелевую" аудиторию, которая удаляет приложение на следующий день)  
Failed Payment Rate = $\frac{Число\ транзакций\ со\ статусом\ failed}{Все\ транзакции}$ (проверка корректности работы платежного шлюза)

### Дизайн эксперимента

$H_0$: разницы между группами A и B нет  
$H_1$: разница есть  
$\alpha$ (вероятность false positive) = 0.05, $\beta$ (вероятность false negative) = 0.2  
**MDE**: базовая конверсия группы A $p_1$ = 0.1, ожидаемая конверсия группы B $p_2$ = 0.125; абсолютный MDE = 2.5%, относительный MDE = 12.5% 

расчёт размера выборки, требуемой для каждой группы
$$n = \frac{\left(Z_{\alpha/2} \cdot \sqrt{2 \cdot \bar{p}(1 - \bar{p})} + Z_{\beta} \cdot \sqrt{p_1(1 - p_1) + p_2(1 - p_2)}\right)^2}{(p_2 - p_1)^2}$$

$p_1 = 0.10$ (базовая конверсия).  
$p_2 = 0.125$ (ожидаемая конверсия).  
$\bar{p} = \frac{p_1 + p_2}{2} = 0.1125$.  
$Z_{\alpha/2} \approx 1.96$ (для $\alpha = 0.05$).  
$Z_{\beta} \approx 0.84$ (для мощности $80\%$).

In [4]:
import random
import os
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [5]:
(1.96 * np.sqrt(2 * 0.1125 * (1 - 0.1125)) + 0.84 * np.sqrt(0.1 * 0.9 + 0.125 * 0.875)) ** 2 / (0.025) ** 2

2503.7036776820173

всего в датасете 12 000 пользователей - достаточно

### Статистический анализ

#### Выгружаем данные

In [6]:
load_dotenv()

DB_USER = "postgres"
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "product_analytics"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

np.random.seed(42)
random.seed(42)

engine = create_engine(DATABASE_URL)

,user_id,group_id,viewed_paywall,purchased,revenue
0,cae3d770-0d0a-4fe5-8fc0-406c08a19952,B,0,0,0.0
1,d1e5fa38-ec6a-4d75-b9bb-fcbdc7c7e527,A,1,0,0.0
2,95ee182b-1f84-426d-b617-d053dd7fadb3,A,1,0,0.0
3,67e9374a-71c4-4097-b715-781b85c20ad4,B,0,0,0.0
4,e4d58056-5e9f-4351-a70e-12ca804a91f8,B,1,0,0.0
